# Lab 11: Government Data APIs

*INFO 4617/5617 · Week 11 · Chapter 11, [Government Data APIs](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html)*

This week every class is live coding: one lab in three parts, one a day. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in. After each step, a **💬 Discuss** box holds questions to talk over before the next one; click it to open it.

We'll use four public APIs, and each one shows an idea that holds for APIs in general:
- **Part 1, Monday:** the **Census Bureau**, and *what can I ask for?*;
- **Part 2, Wednesday:** **FRED**, the Federal Reserve Bank of St. Louis's data service, and *let the server compute*; then the **Energy Information Administration (EIA)**, and *let the server filter*;
- **Part 3, Friday:** **Congress.gov**, and *follow the links*; then a join of two of the APIs.

Along the way, short ⏱ microsprints ask you to change one thing, alone or in pairs, and share what you find.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. Request four free keys and set them as environment variables before you start Jupyter (*Missing Manual* Chapter 34 shows how):
- `CENSUS_KEY`, from the [Census Bureau](https://api.census.gov/data/key_signup.html), for Monday; click the activation link in its email before you use it;
- `FRED_KEY`, from [FRED](https://fred.stlouisfed.org/docs/api/api_key.html), and `EIA_KEY`, from the [EIA](https://www.eia.gov/opendata/register.php), for Wednesday;
- `DATAGOV_KEY`, from [api.data.gov](https://api.data.gov/signup/), for Friday's Congress.gov.

Each part reads only the keys it needs.

On Wednesday and Friday, run every cell above where we stopped (**Run → Run All Above Selected Cell**), or start at that day's setup step.

# Part 1 — Monday: the Census, and what can I ask for?

Every API answers only the questions it was built for, in its own words. Before we can ask the Census Bureau anything, we have to find out what it can tell us (its *variables*) and about what (its *geographies*). The Census API answers both questions itself, in catalogs we can request like any other data. Finding out what an API offers, from the API, is called *discovery*.

## Step 0 — Set up, and keep the key out of the notebook (5 minutes)

An API key identifies you, counts your requests, and is yours alone. A key typed into a notebook travels with the notebook: into a GitHub repository, a class submission, a classmate's laptop. So read the key from an environment variable, as the chapter's ["API Key Management"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#api-key-management) section does. The chapter's setup cell prints whether each of its four keys is set; [`os.environ["NAME"]`](https://docs.python.org/3/library/os.html#os.environ) goes a step further in one line, since a missing key raises a `KeyError` that names it. (The chapter also shows a second way, a `.gitignore`d `api_keys.json` file.) Today we need only `CENSUS_KEY`. Define the User-Agent too: these APIs ask for a key, but every request still says who sent it, in the book's form.

In [ ]:
# Import os, requests, pandas (as pd), and matplotlib.pyplot (as plt)
# Define HEADERS, whose "User-Agent" is "WebDataScience/1.0 (your-email@colorado.edu)", with your email
# Read CENSUS_KEY into census_key with os.environ[...]; never print the key
# Your code here

<details>
<summary>💬 Discuss</summary>

- In week 10 we told Wikipedia who we are with a User-Agent. The Census Bureau also asks for a key: what can a key do that a User-Agent can't?
- Where could a key typed into this notebook end up: a commit, a screenshot, a shared notebook? Which of those could you take back?
- If `CENSUS_KEY` isn't set, what does this cell do? Why is that better than finding out at the first request?

</details>

## Step 1 — What can I ask about? The variable catalog (7 minutes)

The [American Community Survey](https://www.census.gov/data/developers/data-sets/acs-5year.html)'s thousands of variables come in groups (the chapter's ["What Can I Ask For? Variables"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#what-can-i-ask-for-variables) searches all 28,475 of them, about 10 MB; we'll ask for one group's catalog, which is much smaller). Group `B25034` is "Year Structure Built", and the group's catalog is itself an API: send it a GET with [`requests.get()`](https://requests.readthedocs.io/en/stable/api/#requests.get) and our `HEADERS`, and read the answer with `.json()`. Print only the estimates, the codes ending in `E`: the chapter skips the margins of error (`M`), but the group also holds annotation variables (`EA`, `MA`) that would triple the list. Browse all the groups at [api.census.gov/data/2024/acs/acs5/groups.html](https://api.census.gov/data/2024/acs/acs5/groups.html).

In [ ]:
# Request "https://api.census.gov/data/2024/acs/acs5/groups/B25034.json" with HEADERS; read the JSON into variables
# Loop over sorted(variables["variables"].items()), and print each code that ends in "E" with its label
# Your code here

<details>
<summary>💬 Discuss</summary>

- What do the parts of `B25034_002E` tell you: the table, the line, and the final letter? Why would an estimate need an annotation variable beside it?
- In week 6, The Numbers hid its table among decoys. How is picking a variable from this catalog like that, and how is it different?
- Every cell today assumes its request works. If the Census Bureau answered a 429, or a web page with status 200, which line would fail, and would week 10's `raise_for_status()` have caught either?

</details>

## Step 2 — Where can I ask about? The geography catalog (6 minutes)

The variables are half of a question; the other half is a place. The Census divides the country into nested geographies (states, counties, places, tracts, and more), and the dataset's [geography catalog](https://api.census.gov/data/2024/acs/acs5/geography.html) lists every level it can answer for, and what each one *requires*: the larger geography you must name with it. It is an API too, at `geography.json`. Print the requirements of four levels, by their summary-level codes, as the chapter's ["What Can I Ask For? Geographies"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#what-can-i-ask-for-geographies) does: a state is `040`, a county `050`, a census tract `140`, and a place, such as a city, `160`.

In [ ]:
# Request "https://api.census.gov/data/2024/acs/acs5/geography.json" with HEADERS; levels: the JSON's "fips" list
# Print the "name" and .get("requires") of each level whose "geoLevelDisplay" is "040", "050", "140", or "160"
# Your code here

<details>
<summary>💬 Discuss</summary>

- A city is a `place`. What else must a request about a place name, and why would a `tract` need a county as well as a state?
- Steps 1 and 2 asked the API what it knows before asking it anything. Without the two catalogs, how would you have found `B25034` or `place`?
- Which catalog would you check again next year, when the Census publishes the 2023 estimates, and why might its answer change?

</details>

## Step 3 — Ask about one place (5 minutes)

Step 2 says a place needs its state. Places are named by [FIPS codes](https://www.census.gov/library/reference/code-lists/ansi.html): Colorado is state `08`, and Boulder is place `07850`. As in the chapter's ["Querying for a Place"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#querying-for-a-place), ask for the place's name and the first few `B25034` estimates, with `get`, `for`, and `in` parameters, and the key. The answer is a list of lists whose first row is the header, and every value is a string, so make a DataFrame and turn the numbers into numbers with [`pd.to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html). Keep the names `census_base` (the chapter's) and `numeric_cols`: we use them again.

In [ ]:
# census_base = "https://api.census.gov/data/2024/acs/acs5"
# params: get="NAME,B25034_001E,B25034_002E,B25034_003E,B25034_004E,B25034_005E", for="place:07850", in="state:08", key=census_key
# Request census_base with params and HEADERS; make df from rows 1 onward, with row 0 as the columns
# numeric_cols: the columns that start with "B25034"; convert them with pd.to_numeric, and show df
# Your code here

<details>
<summary>💬 Discuss</summary>

- In week 5, `urlparse()` took apart a Census URL just like this request's. Which parameters name the variables, which the place, and which must never appear in a printed error?
- In week 7, the Wayback Machine's CDX API sent the same shape: a list of lists with a header row. Why might an API send data this way rather than as nested dictionaries?
- Why do the numbers arrive as strings, and why must a FIPS code like `08` stay one?

</details>

## Step 4 — Compare two places (12 minutes)

Several places can go in one request: `place:07850,45970` is Boulder and Longmont (the chapter's ["Comparing Places"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#comparing-places)). Change one parameter and ask again, then [plot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.bar.html) the two cities' housing by decade built, side by side. Leave the total out of the chart, since it would dwarf the decades, and label the decades; the chapter takes its labels from the catalog, and we'll type four.

In [ ]:
# Set params["for"] to "place:07850,45970" and request again
# Build df_compare as in Step 3, convert numeric_cols, and show it
# Your code here

In [ ]:
# df_plot: df_compare indexed by NAME, the decade columns only (numeric_cols[1:]), transposed; name its rows by decade
# On plt.subplots(), plot df_plot as grouped bars, with a title, axis labels, and a legend titled "City"
# Your code here

### ⏱ Microsprint (6 minutes, alone or in pairs)

A short version of Additional Exercise 1. Cities differ in size, so raw counts don't compare fairly; shares do.

**Do:** ask for the total (`B25034_001E`) and the three decades since 2000 (`_002E` to `_004E`) for five Colorado places, and compute the share of each city's housing built in 2000 or later. The key uses Boulder, Longmont, Fort Collins (`27425`), Lakewood (`43000`), and Castle Rock (`12415`); look up others in the [FIPS code lists](https://www.census.gov/library/reference/code-lists/ansi.html).
**Share:** the city with the largest share, the one with the smallest, and the two shares.

In [ ]:
# Microsprint: set params["get"] to NAME and B25034_001E to B25034_004E, params["for"] to five places, and request
# df_five: indexed by NAME, the four counts converted with pd.to_numeric
# since_2000: 002E + 003E + 004E; share: since_2000 / 001E; show share, largest first
# Your code here

<details>
<summary>💬 Discuss</summary>

- Which of your five cities built the most of its housing since 2000, and which the least? What might explain the difference?
- Longmont out-built Boulder in every decade the chart shows. Is that a fair comparison, and what does the share add?
- These are 2020–2024 five-year estimates. What does that mean for a count labeled "2020 or later"?

</details>

## Step 5 — Sanity-check a number (8 minutes)

A variable with the right-sounding label can still be the wrong one. The chapter's ["Querying for a Place"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#querying-for-a-place) checks its answer against a value it can judge, the estimate on data.census.gov's table. Do the same before you trust a pipeline. Ask for Boulder's median household income, [`B19013_001E`](https://api.census.gov/data/2024/acs/acs5/groups/B19013.html). Does it look like a city's median income: tens of thousands of dollars, not 7 or 70 million?

In [ ]:
# params: get="NAME,B19013_001E", for="place:07850", in="state:08", key=census_key; request it
# Print the city and its median household income, with a dollar sign and commas
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

Check the step's result another way. Every estimate has a [margin of error](https://www.census.gov/content/dam/Census/library/publications/2020/acs/acs_general_handbook_2020_ch07.pdf), the variable with the same code ending in `M`; the American Community Survey gives it at 90% confidence.

**Do:** ask for `B19013_001E` and `B19013_001M` for Boulder and Longmont, and compute each city's interval: the estimate minus and plus its margin.
**Share:** the two intervals, and whether they overlap.

In [ ]:
# Microsprint: set params["get"] to NAME, B19013_001E, and B19013_001M, params["for"] to "place:07850,45970", and request
# incomes: indexed by NAME, the two columns converted; add low (estimate - margin) and high (estimate + margin)
# Show incomes: do the two intervals overlap?
# Your code here

<details>
<summary>💬 Discuss</summary>

- Does Boulder's median income look right? What did you compare it with?
- Did the two intervals overlap? If they had, what could you no longer say about which city's households earn more?
- Which checks like this one could a pipeline run on its own, every time it runs?

</details>

## Step 6 — When there's no estimate (6 minutes)

Eighty-three of Colorado's 482 places have no estimate of median household income in the 2020–2024 survey (72 did in 2018–2022): too few households answered for the Census Bureau to compute one. The chapter's ["Every Place in a State, and a Number That Isn't"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-11-government.html#every-place-in-a-state-and-a-number-that-isnt) counts them. The API doesn't leave the value out. It sends a number that can't be an income, `-666666666`, and `-222222222` for its margin of error; the Bureau's [notes on estimate and annotation values](https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html) list what each code means. Ask for the population (`B01003_001E`), the median income, and its margin for Boulder and three small towns: Bonanza (`07571`), Campo (`11645`), and Carbonate (`12030`). Look at what comes back, then set every negative value aside with [`mask()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mask.html), which turns it into a missing value, `NaN`.

In [ ]:
# params: get="NAME,B01003_001E,B19013_001E,B19013_001M", for="place:07850,07571,11645,12030", in="state:08", key=census_key
# Request it; towns: indexed by NAME, the three columns converted with pd.to_numeric; show towns
# Your code here

In [ ]:
# Print the mean of the median-income column as it is; then mask every value below zero, and print the mean again
# Your code here

<details>
<summary>💬 Discuss</summary>

- What would a chart of median income have shown with `-666666666` left in? Would you have noticed?
- Why might an API send a code like this rather than leave the value empty? What does a missing value lose that the code keeps?
- Campo has an estimate this time, with a margin of error almost as large. Is that more useful than Bonanza's code, or less?
- Carbonate has no residents, yet it has a row. What does the Census count as a place, and what does that mean for a table of "every place"?

</details>

## End of Part 1

On Wednesday, Part 2 asks FRED to compute for us, then the EIA to filter for us; on Friday, Part 3 follows Congress.gov's links, and joins two of the week's APIs. Each part is added to this notebook before its day. Download the new copy, and start at its setup step: it imports and reads the keys again, so it doesn't need Monday's cells to have run.